# CNMF-E `min_corr` / `min_pnr` selection notebook

这个 notebook 只做一件事：从已有的 CaImAn `.mmap` 中裁出一个你在 FIJI 里选好的 ROI，计算 CNMF-E 初始化使用的 correlation image 和 PNR image，然后用几种可视化辅助你选择 `min_corr` 和 `min_pnr`。

它不会跑 CNMF-E，不会写 `data/`，也不会改模型文件。

## How to use

1. 在下面第一个 code cell 里确认 `MMAP_PATH`，以及 FIJI ROI 的 `ROI_X / ROI_Y / ROI_WIDTH / ROI_HEIGHT`。
2. 从上到下运行到最后的 interactive dashboard。
3. 在 dashboard 里拖动 `min_corr` 和 `min_pnr`，同时看 ROI overlay、candidate seeds、histogram 阈值线、PNR-vs-correlation density 和 candidate fraction grid。
4. dashboard 旁边的 YAML box 会实时更新，最后把选好的两个值复制回 `config/cnmfe.yaml`。

FIJI 坐标约定：`X coordinate` 是列方向，`Y coordinate` 是行方向。CaImAn 的 image shape 是 `(height, width)`，所以这里会用 `image[y:y+height, x:x+width]`。

In [ ]:
# User inputs
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    # If this notebook is opened from notebook/, use the repository root.
    ROOT = ROOT.parent

MMAP_PATH = ROOT / "data" / "mmap" / "Y_d1_2692_d2_3548_d3_1_order_C_frames_1788.mmap"

# FIJI rectangle ROI from the screenshot.
ROI_X = 785
ROI_Y = 1940
ROI_WIDTH = 300
ROI_HEIGHT = 300

# CNMF-E initialization spatial scale. Keep this aligned with config/cnmfe.yaml.
GSIG = (3, 3)

# Official CaImAn demo subsamples to about 1000 frames for correlation_pnr when T is large.
TARGET_SAMPLE_FRAMES = 1000

# Starting point copied from the CaImAn CNMF-E demo and your current config.
MIN_CORR = 0.85
MIN_PNR = 12.0

print(f"repo root: {ROOT}")
print(f"mmap: {MMAP_PATH}")

## Imports and loader

This uses the project's mmap filename parser, so the notebook reads exactly the same encoded CaImAn mmap style as the scripts.

In [ ]:
import importlib.util
import os
import warnings

os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")

import matplotlib.pyplot as plt
import numpy as np
from scipy import ndimage as ndi

import caiman as cm

try:
    import cv2
    cv2.setNumThreads(0)
except Exception:
    pass

try:
    import ipywidgets as widgets
    from IPython.display import display
    HAVE_WIDGETS = True
except Exception:
    HAVE_WIDGETS = False

spec = importlib.util.spec_from_file_location("cm2_mmap", ROOT / "src" / "mmap.py")
cm2_mmap = importlib.util.module_from_spec(spec)
spec.loader.exec_module(cm2_mmap)

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

warnings.filterwarnings("ignore", category=RuntimeWarning)

## Load ROI from mmap

The full movie is not loaded into RAM. This cell creates a memmap view, then copies only the sampled ROI frames into memory.

In [ ]:
Yr, dims, T = cm2_mmap.load_memmap_movie(MMAP_PATH)
height, width = dims
images = np.reshape(Yr.T, (T, height, width), order="F")

x0 = int(ROI_X)
y0 = int(ROI_Y)
x1 = min(x0 + int(ROI_WIDTH), width)
y1 = min(y0 + int(ROI_HEIGHT), height)

if x0 < 0 or y0 < 0 or x0 >= width or y0 >= height or x1 <= x0 or y1 <= y0:
    raise ValueError(f"ROI is outside movie bounds: movie={(height, width)}, roi={(x0, y0, x1, y1)}")

sample_step = max(T // int(TARGET_SAMPLE_FRAMES), 1)
frame_idx = np.arange(0, T, sample_step, dtype=int)
if frame_idx.size > int(TARGET_SAMPLE_FRAMES):
    frame_idx = frame_idx[: int(TARGET_SAMPLE_FRAMES)]

roi_movie = np.asarray(images[frame_idx, y0:y1, x0:x1], dtype=np.float32)

print(f"full movie: T={T}, height={height}, width={width}")
print(f"ROI: x={x0}:{x1}, y={y0}:{y1}, shape={roi_movie.shape}")
print(f"sampled frames: {frame_idx.size} / {T}, step={sample_step}")
print(f"ROI memory: {roi_movie.nbytes / 1024**2:.1f} MiB")

## Precompute ROI summaries

These arrays are used by the interactive dashboard below. No static diagnostic figures are shown here, so threshold selection happens in one place.

In [ ]:
roi_mean = roi_movie.mean(axis=0)
roi_max = roi_movie.max(axis=0)
roi_std = roi_movie.std(axis=0)
print(f"ROI summaries ready: mean/max/std shape = {roi_mean.shape}")

## Compute correlation and PNR images

This mirrors the official CaImAn CNMF-E demo: compute `correlation_pnr` on sampled frames using `gSig`. The output maps are what `method_init: corr_pnr` uses to decide candidate seed pixels.

In [ ]:
correlation_image, pnr_image = cm.summary_images.correlation_pnr(
    roi_movie,
    gSig=int(GSIG[0]),
    swap_dim=False,
)

valid = np.isfinite(correlation_image) & np.isfinite(pnr_image)
corr_values = correlation_image[valid]
pnr_values = pnr_image[valid]

print(f"correlation range: {np.nanmin(correlation_image):.3f} to {np.nanmax(correlation_image):.3f}")
print(f"PNR range: {np.nanmin(pnr_image):.3f} to {np.nanmax(pnr_image):.3f}")

## Interactive threshold dashboard

Use this as the main workspace for choosing `min_corr` and `min_pnr`. The sliders update while you drag. The dashboard shows the current threshold on the histograms, the correlation-vs-PNR density, the candidate mask, approximate seed locations, and a copy-ready YAML snippet.

A good setting usually keeps plausible neural structures in the seed overlay while avoiding broad vessel/background regions. For CNMF-E initialization, being slightly permissive is usually safer than being too strict, because downstream QC can remove false positives but cannot recover missed seeds.

In [ ]:
if HAVE_WIDGETS:
    from io import BytesIO

    slider_style = {"description_width": "90px"}
    slider_layout = widgets.Layout(width="430px")
    image_layout = widgets.Layout(width="100%", min_height="980px")

    corr_plot_min = 0.0
    corr_plot_max = 1.0
    pnr_plot_max = max(25.0, float(np.nanpercentile(pnr_values, 99.7)))

    corr_counts, corr_edges = np.histogram(corr_values, bins=90, range=(corr_plot_min, corr_plot_max))
    pnr_counts, pnr_edges = np.histogram(np.clip(pnr_values, 0, pnr_plot_max), bins=90, range=(0, pnr_plot_max))
    joint_counts, joint_corr_edges, joint_pnr_edges = np.histogram2d(
        corr_values,
        np.clip(pnr_values, 0, pnr_plot_max),
        bins=(90, 90),
        range=((corr_plot_min, corr_plot_max), (0, pnr_plot_max)),
    )
    joint_image = np.log1p(joint_counts.T)

    corr_grid = np.round(np.linspace(0.65, 0.95, 13), 2)
    pnr_grid = np.round(np.linspace(6.0, max(18.0, float(np.nanpercentile(pnr_values, 97.5))), 13), 1)
    candidate_fraction_grid = np.zeros((len(pnr_grid), len(corr_grid)), dtype=float)
    for i, pnr_thr in enumerate(pnr_grid):
        for j, corr_thr in enumerate(corr_grid):
            candidate_fraction_grid[i, j] = (
                ((correlation_image >= corr_thr) & (pnr_image >= pnr_thr) & valid).sum() / valid.sum()
            )

    def fig_to_png_bytes(fig, dpi=105):
        buf = BytesIO()
        fig.savefig(buf, format="png", dpi=dpi, bbox_inches="tight")
        plt.close(fig)
        return buf.getvalue()

    def candidate_seed_points(min_corr=MIN_CORR, min_pnr=MIN_PNR, min_distance=None, max_points=500):
        if min_distance is None:
            min_distance = max(3, int(GSIG[0]))
        mask = (correlation_image >= min_corr) & (pnr_image >= min_pnr) & valid
        score = np.where(mask, correlation_image * pnr_image, 0.0)
        size = 2 * int(min_distance) + 1
        local_max = (score == ndi.maximum_filter(score, size=size, mode="nearest")) & (score > 0)
        coords = np.argwhere(local_max)
        if coords.size == 0:
            return coords, score, mask
        order = np.argsort(score[coords[:, 0], coords[:, 1]])[::-1]
        coords = coords[order[:max_points]]
        return coords, score, mask

    def draw_threshold_dashboard(min_corr, min_pnr, max_seeds):
        coords, score, mask = candidate_seed_points(min_corr, min_pnr, max_points=max_seeds)
        candidate_pixels = int(mask.sum())
        candidate_fraction = candidate_pixels / int(valid.sum())
        corr_pass = float((corr_values >= min_corr).mean())
        pnr_pass = float((pnr_values >= min_pnr).mean())
        both_pass = float(((corr_values >= min_corr) & (pnr_values >= min_pnr)).mean())

        fig = plt.figure(figsize=(16.5, 12.0), constrained_layout=True)
        fig.set_constrained_layout_pads(w_pad=0.015, h_pad=0.015, wspace=0.02, hspace=0.02)
        gs = fig.add_gridspec(3, 3, height_ratios=[1.85, 0.95, 0.8])

        ax_overlay = fig.add_subplot(gs[0, 0])
        ax_corr_map = fig.add_subplot(gs[0, 1])
        ax_pnr_map = fig.add_subplot(gs[0, 2])
        ax_corr_hist = fig.add_subplot(gs[1, 0])
        ax_pnr_hist = fig.add_subplot(gs[1, 1])
        ax_joint = fig.add_subplot(gs[1, 2])
        ax_score = fig.add_subplot(gs[2, 0])
        ax_grid = fig.add_subplot(gs[2, 1])
        ax_text = fig.add_subplot(gs[2, 2])

        ax_overlay.imshow(roi_mean, cmap="gray")
        ax_overlay.imshow(np.ma.masked_where(~mask, mask), cmap="autumn", alpha=0.45)
        if coords.size:
            ax_overlay.scatter(coords[:, 1], coords[:, 0], s=13, facecolors="none", edgecolors="#00d4ff", linewidths=0.75)
        ax_overlay.set_title(f"ROI + threshold mask + seeds\nseeds={len(coords):,}, candidate pixels={candidate_pixels:,}")

        im_corr = ax_corr_map.imshow(
            correlation_image,
            cmap="inferno",
            vmin=np.nanpercentile(correlation_image, 1),
            vmax=np.nanpercentile(correlation_image, 99.5),
        )
        ax_corr_map.contour(mask.astype(float), levels=[0.5], colors="#00d4ff", linewidths=0.5)
        ax_corr_map.set_title(f"correlation map\nthreshold >= {min_corr:.2f}")
        fig.colorbar(im_corr, ax=ax_corr_map, fraction=0.035, pad=0.015)

        im_pnr = ax_pnr_map.imshow(
            pnr_image,
            cmap="inferno",
            vmin=np.nanpercentile(pnr_image, 1),
            vmax=np.nanpercentile(pnr_image, 99.5),
        )
        ax_pnr_map.contour(mask.astype(float), levels=[0.5], colors="#00d4ff", linewidths=0.5)
        ax_pnr_map.set_title(f"PNR map\nthreshold >= {min_pnr:.1f}")
        fig.colorbar(im_pnr, ax=ax_pnr_map, fraction=0.035, pad=0.015)

        corr_centers = 0.5 * (corr_edges[:-1] + corr_edges[1:])
        ax_corr_hist.fill_between(corr_centers, corr_counts, color="#2f6f9f", alpha=0.82, step="mid")
        ax_corr_hist.axvline(min_corr, color="#c43c39", lw=2)
        ax_corr_hist.set_title(f"correlation histogram\npass={corr_pass:.2%}")
        ax_corr_hist.set_xlabel("correlation")
        ax_corr_hist.set_ylabel("pixels")
        ax_corr_hist.set_xlim(corr_plot_min, corr_plot_max)

        pnr_centers = 0.5 * (pnr_edges[:-1] + pnr_edges[1:])
        ax_pnr_hist.fill_between(pnr_centers, pnr_counts, color="#b7791f", alpha=0.82, step="mid")
        ax_pnr_hist.axvline(min_pnr, color="#c43c39", lw=2)
        ax_pnr_hist.set_title(f"PNR histogram\npass={pnr_pass:.2%}")
        ax_pnr_hist.set_xlabel(f"PNR, clipped at {pnr_plot_max:.1f}")
        ax_pnr_hist.set_ylabel("pixels")
        ax_pnr_hist.set_xlim(0, pnr_plot_max)

        ax_joint.imshow(
            joint_image,
            origin="lower",
            aspect="auto",
            extent=(corr_plot_min, corr_plot_max, 0, pnr_plot_max),
            cmap="viridis",
        )
        ax_joint.axvline(min_corr, color="#c43c39", lw=2)
        ax_joint.axhline(min_pnr, color="#c43c39", lw=2)
        ax_joint.set_title(f"PNR vs correlation density\nupper-right={both_pass:.2%}")
        ax_joint.set_xlabel("correlation")
        ax_joint.set_ylabel("PNR")

        score_vmax = np.nanpercentile(score[score > 0], 99) if np.any(score > 0) else 1
        im_score = ax_score.imshow(score, cmap="magma", vmin=0, vmax=score_vmax)
        if coords.size:
            ax_score.scatter(coords[:, 1], coords[:, 0], s=10, facecolors="none", edgecolors="#00d4ff", linewidths=0.7)
        ax_score.set_title("thresholded corr x PNR score")
        fig.colorbar(im_score, ax=ax_score, fraction=0.046, pad=0.04)

        im_grid = ax_grid.imshow(
            candidate_fraction_grid,
            origin="lower",
            aspect="auto",
            extent=(corr_grid.min(), corr_grid.max(), pnr_grid.min(), pnr_grid.max()),
            cmap="viridis",
        )
        ax_grid.scatter([min_corr], [min_pnr], s=70, facecolors="none", edgecolors="#ff3b30", linewidths=2)
        ax_grid.set_title("candidate pixel fraction grid\nred circle = current threshold")
        ax_grid.set_xlabel("min_corr")
        ax_grid.set_ylabel("min_pnr")
        fig.colorbar(im_grid, ax=ax_grid, fraction=0.046, pad=0.04, format="%.2f")

        ax_text.axis("off")
        text = (
            "Current choice\n"
            f"min_corr: {min_corr:.2f}\n"
            f"min_pnr:  {min_pnr:.1f}\n\n"
            "Pixel gates\n"
            f"corr pass: {corr_pass:.2%}\n"
            f"PNR pass:  {pnr_pass:.2%}\n"
            f"both pass: {both_pass:.2%}\n\n"
            "Approx seed overlay\n"
            f"seeds shown: {len(coords):,}\n"
            f"candidate pixels: {candidate_pixels:,}\n"
            f"candidate fraction: {candidate_fraction:.2%}\n\n"
            "Copy to config/cnmfe.yaml\n"
            "params:\n"
            "  init:\n"
            f"    min_corr: {min_corr:.3g}\n"
            f"    min_pnr: {min_pnr:.3g}"
        )
        ax_text.text(0.0, 1.0, text, va="top", ha="left", family="monospace", fontsize=10)

        for ax in (ax_overlay, ax_corr_map, ax_pnr_map):
            ax.set_xticks([])
            ax.set_yticks([])

        for ax in (ax_score,):
            ax.set_xlabel("x within ROI")
            ax.set_ylabel("y within ROI")

        return fig_to_png_bytes(fig)

    min_corr_slider = widgets.FloatSlider(
        value=MIN_CORR,
        min=0.50,
        max=0.98,
        step=0.01,
        description="min_corr",
        continuous_update=True,
        readout_format=".2f",
        style=slider_style,
        layout=slider_layout,
    )
    min_pnr_slider = widgets.FloatSlider(
        value=MIN_PNR,
        min=2.0,
        max=pnr_plot_max,
        step=0.5,
        description="min_pnr",
        continuous_update=True,
        readout_format=".1f",
        style=slider_style,
        layout=slider_layout,
    )
    max_seeds_slider = widgets.IntSlider(
        value=500,
        min=50,
        max=1500,
        step=50,
        description="max seeds",
        continuous_update=True,
        style=slider_style,
        layout=slider_layout,
    )

    dashboard_img = widgets.Image(format="png", layout=image_layout)
    yaml_box = widgets.Textarea(
        description="YAML",
        layout=widgets.Layout(width="430px", height="92px"),
        style=slider_style,
    )

    def update_yaml_box():
        yaml_box.value = (
            "params:\n"
            "  init:\n"
            f"    min_corr: {min_corr_slider.value:.3g}\n"
            f"    min_pnr: {min_pnr_slider.value:.3g}\n"
        )

    def render_dashboard(_=None):
        update_yaml_box()
        dashboard_img.value = draw_threshold_dashboard(
            min_corr_slider.value,
            min_pnr_slider.value,
            max_seeds_slider.value,
        )

    for control in (min_corr_slider, min_pnr_slider, max_seeds_slider):
        control.observe(render_dashboard, names="value")

    display(
        widgets.VBox(
            [
                widgets.HBox([min_corr_slider, min_pnr_slider]),
                widgets.HBox([max_seeds_slider, yaml_box]),
                dashboard_img,
            ]
        )
    )
    render_dashboard()
else:
    print("ipywidgets is not available. Edit MIN_CORR and MIN_PNR in the first cell, then rerun this dashboard cell.")